Importing main script

In [21]:
import sys
sys.path.append('../../src')

# Import the module to be reloaded
import Simple_Model_r0
import importlib
importlib.reload(Simple_Model_r0)

# Import everything from the r<eloaded module
from Simple_Model_r0 import * 

### Example usage

#### Example of Node

In [2]:
geom = Geometry(
    # Height and length of the heat exchanger
    width=1,
    length=1,
    # Pipe dimensions
    pipe_outer_diameter=0.0603, 
    pipe_wall_thickness=0.005,
    # Arrengement dimensions
    arrangement="Inline", 
    transverse_pitch=0.04, 
    longitudinal_pitch=0.04
    )

node1 = Node(
        Geometry=geom,
        x_pos=0,
        node_length=0.1, #m
        T_hot_init=343, # C 
        T_cold_init=200, # C 
        P_hot_init=158e5, # Pa
        P_cold_init=1e5, # Pa
        m_dot_hot=0.1, # kg/s
        m_dot_cold=10, # kg/s
        roughness=0.000007,
        is_boundary=True
    )
node1.Fluid_hot.update(Input.pressure(158e5), Input.quality(0.98))

print(node1.Fluid_hot.quality)
    
Delta_H = node1.overall_HTC()[0]
print(f"Initial Delta_H: {Delta_H}")
length = 0 

for i in range(100):
    length = length + node1.node_length
    node1.Fluid_hot.update(Input.enthalpy(node1.Fluid_hot.enthalpy + Delta_H), Input.pressure(158e5))
    Delta_H = node1.overall_HTC()[0]
    print(f"-{i}- length: {length:.1f} x: {node1.Fluid_hot.quality}, Delta_H: {Delta_H}")  

0.98
Initial Delta_H: -1113.039951110836
-0- length: 0.1 x: 0.978822517396386, Delta_H: -1113.0611583824166
-1- length: 0.2 x: 0.9776450123576446, Delta_H: -1113.0808760693446
-2- length: 0.3 x: 0.9764674864596041, Delta_H: -1113.0992569952753
-3- length: 0.4 x: 0.9752899411164208, Delta_H: -1113.116431455967
-4- length: 0.5 x: 0.9741123776044119, Delta_H: -1113.1325114377341
-5- length: 0.6 x: 0.9729347970814242, Delta_H: -1113.147593892936
-6- length: 0.7 x: 0.9717572006027394, Delta_H: -1113.161763314559
-7- length: 0.8 x: 0.9705795891342535, Delta_H: -1113.1750937824488
-8- length: 0.9 x: 0.9694019635634934, Delta_H: -1113.18765060614
-9- length: 1.0 x: 0.9682243247088967, Delta_H: -1113.1994916560782
-10- length: 1.1 x: 0.9670466733276787, Delta_H: -1113.2106684515536
-11- length: 1.2 x: 0.965869010122552, Delta_H: -1113.221227056805
-12- length: 1.3 x: 0.9646913357474992, Delta_H: -1113.2312088244942
-13- length: 1.4 x: 0.9635136508127559, Delta_H: -1113.2406510167134
-14- length

In [3]:
geom = Geometry(
    # Height and length of the heat exchanger
    width=1,
    length=1,
    # Pipe dimensions
    pipe_outer_diameter=0.0603, 
    pipe_wall_thickness=0.005,
    # Arrengement dimensions
    arrangement="Inline", 
    transverse_pitch=0.04, 
    longitudinal_pitch=0.04
    )

node1 = Node(
        Geometry=geom,
        x_pos=0,
        node_length=0.1, #m
        T_hot_init=343, # C 
        T_cold_init=200, # C 
        P_hot_init=158e5, # Pa
        P_cold_init=1e5, # Pa
        m_dot_hot=0.1, # kg/s
        m_dot_cold=10, # kg/s
        roughness=0.000007,
        is_boundary=True
    )

node1.Fluid_hot.update(Input.pressure(158e5), Input.enthalpy(node1.Fluid_hot.dew_point_at_pressure(158e5).enthalpy))

print(node1.overall_HTC())

[np.float64(-1085.9406817356664), np.float64(46.96075797971145)]


#### Example of fluid class

In [ ]:
geom = Geometry(
    # Height and length of the heat exchanger
    width=1,
    length=1,
    # Pipe dimensions
    pipe_outer_diameter=0.025, 
    pipe_wall_thickness=0.002,
    # Arrengement dimensions
    arrangement="Staggered", 
    transverse_pitch=0.04, 
    longitudinal_pitch=0.04
    )

# Define the fluid (water in this case)
water = Fluid(FluidsList.Water,10)

# Set the state 
water.update(Input.enthalpy(430340.09), Input.pressure(158e5))  # 1 bar = 100000 Pa

water.set_geometry(geom, 'internal')
print(water.units_system)
water.dynamic_viscosity

print(water.liquid_phase().enthalpy)
print(water.vapor_phase().enthalpy)

water.critical_pressure

SIWithCelsius


ValueError: Fluid(type=Water, T=99.84368203717378 C, P=15800000.0 Pa, , H=430340.09 J/kg, x=None) is not in a two-phase state.

In [15]:
pf.PyFluidsConfig.units_system

SIWithCelsius

In [ ]:
from enum import StrEnum
import numpy as np


class arrangementType(StrEnum):
    '''Enum for tube arrangement types'''
    Inline = "Inline"
    Staggered = "Staggered"
    Single_Tube = "Single_Tube"
    Single_Row = "Single_Row"

class Geometry:
    '''Class representing the geometry of the heat exchanger with given dimensions'''
    class _Tube:
        '''Inner class representing tube geometry'''
        def __init__(self):
            pass
        def set_geometry(self,outer_diameter,wall_thickness):
            self.outer_diameter = outer_diameter
            self.wall_thickness = wall_thickness
            
            # Derived properties
            self.inner_diameter = self.outer_diameter - 2 * self.wall_thickness
            self.inner_perimeter= np.pi * self.inner_diameter
            self.outer_perimeter= np.pi * self.outer_diameter
            
    class _Duct:
        '''Inner class representing duct geometry'''
        def __init__(self):
            pass
        def set_geometry(self,width,height):
            self.a = height
            self.b  = width
            
            # Derived properties
            self.area   = self.a * self.b
            
    class _Fin:
        '''Inner class representing fin geometry'''
        def __init__(self):
            pass
        def set_geometry(self,average_fin_thickness,fin_height,fin_spacing):
            self.average_fin_thickness  = average_fin_thickness #delta_r
            self.fin_height             = fin_height #l_r
            self.fin_spacing            = fin_spacing #s_r
    
    class _Bank: 
        '''Inner class representing bank geometry'''
        def __init__(self):
            pass
        def set_geometry(self, number_of_rows,transverse_pitch,longitudinal_pitch,arrangement=arrangementType.Inline):
            self.number_of_rows     = number_of_rows
            self.transverse_pitch   = transverse_pitch
            self.longitudinal_pitch = longitudinal_pitch
            self.arrangement        = arrangement
            
            # Derived properties
            match arrangement:
                case arrangementType.Inline:
                    self.diagonal_pitch = None
                case arrangementType.Staggered:
                    self.diagonal_pitch = np.sqrt(self.longitudinal_pitch**2+(self.transverse_pitch/2)**2)
                case arrangementType.Single_Tube:
                    pass 
                case arrangementType.Single_Row:
                    pass
    
    def __init__(self):
        # Initialize inner classes
        self.Tube = self._Tube()
        self.Duct = self._Duct()
        self.Fin  = self._Fin()
        self.Bank = self._Bank()
        
    
geom = Geometry() 
geom.Duct.set_geometry(5,3)
geom.Duct.area

15